In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.utils import to_categorical

import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
import subprocess
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

subprocess.run(
    'echo "ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true" | debconf-set-selections',
    shell=True
)

result = subprocess.run(
    "apt-get update -qq && apt-get install -y ttf-mscorefonts-installer",
    shell=True,
    capture_output=True,
    text=True
)

print(result.stdout[-1000:])
print(result.stderr[-1000:])

subprocess.run("rm -rf ~/.cache/matplotlib", shell=True)
subprocess.run("fc-cache -f", shell=True, capture_output=True)

for f in fm.findSystemFonts():
    if "times" in f.lower():
        fm.fontManager.addfont(f)

available = {f.name for f in fm.fontManager.ttflist}

print("\nIs 'Times New Roman' actually installed?",
      "Times New Roman" in available)

if "Times New Roman" in available:
    plt.rcParams["font.family"] = "Times New Roman"
else:
    subprocess.run(
        "apt-get install -y fonts-liberation",
        shell=True,
        capture_output=True
    )

    subprocess.run(
        "rm -rf ~/.cache/matplotlib && fc-cache -f",
        shell=True,
        capture_output=True
    )

    for f in fm.findSystemFonts():
        if "liberationserif" in f.lower():
            fm.fontManager.addfont(f)

    plt.rcParams["font.family"] = "Liberation Serif"

    print(
        "Times New Roman unavailable — using Liberation Serif instead."
    )

plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 11
plt.rcParams["figure.titlesize"] = 14

print("\nFinal font:", plt.rcParams["font.family"])

In [ ]:
def save_fig(fig, filename):
    fig.savefig(f"{filename}.pdf", format="pdf", dpi=600, bbox_inches="tight")
    print(f"Saved {filename}.pdf")

In [ ]:
import os


try:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    CACHE_DIR = "/content/drive/MyDrive/cifar10_cache"
except ImportError:
    CACHE_DIR = "./cifar10_cache"

os.makedirs(CACHE_DIR, exist_ok=True)
CIFAR_CACHE_PATH = os.path.join(CACHE_DIR, "cifar10.npz")

if os.path.exists(CIFAR_CACHE_PATH):
    print("Loading CIFAR-10 from Drive cache...")
    cached = np.load(CIFAR_CACHE_PATH)
    x_train, y_train_raw = cached["x_train"], cached["y_train_raw"]
    x_test, y_test_raw = cached["x_test"], cached["y_test_raw"]
else:
    print("No cache found — downloading CIFAR-10 (one-time only)...")
    (x_train, y_train_raw), (x_test, y_test_raw) = keras.datasets.cifar10.load_data()
    np.savez(CIFAR_CACHE_PATH, x_train=x_train, y_train_raw=y_train_raw,
              x_test=x_test, y_test_raw=y_test_raw)
    print(f"Cached to {CIFAR_CACHE_PATH} for future runs.")


x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0


y_train = to_categorical(y_train_raw, num_classes=10)
y_test = to_categorical(y_test_raw, num_classes=10)


print("Training data shape  :", x_train.shape)
print("Training labels shape:", y_train.shape)
print("Testing data shape   :", x_test.shape)
print("Testing labels shape :", y_test.shape)


fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(CLASS_NAMES[int(y_train_raw[i])], fontsize=11)
    ax.axis("off")
fig.suptitle("Sample CIFAR-10 Images")
plt.tight_layout()
save_fig(fig, "fig1_sample_cifar10_images")
plt.show()

In [ ]:
IMG_SIZE = 96


base_model = VGG16(weights="imagenet", include_top=False,
                    input_shape=(IMG_SIZE, IMG_SIZE, 3))


base_model.trainable = False

inputs = keras.Input(shape=(32, 32, 3))
x = layers.Resizing(IMG_SIZE, IMG_SIZE)(inputs)
x = layers.Lambda(lambda img: img * 255.0)(x)

x = layers.Lambda(preprocess_input)(x)
x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(10, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="VGG16_TransferLearning")
model.summary()

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

EPOCHS = 15
BATCH_SIZE = 32

start_time = time.time()
history = model.fit(x_train, y_train,
                     validation_data=(x_test, y_test),
                     epochs=EPOCHS,
                     batch_size=BATCH_SIZE)
train_time_phase1 = time.time() - start_time
print(f"\nPhase-1 (frozen base) training time: {train_time_phase1:.1f} seconds")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history.history["accuracy"], label="Training Accuracy", linewidth=2)
ax.plot(history.history["val_accuracy"], label="Validation Accuracy", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Training and Validation Accuracy")
ax.legend()
plt.tight_layout()
save_fig(fig, "fig2_training_validation_accuracy")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history.history["loss"], label="Training Loss", linewidth=2)
ax.plot(history.history["val_loss"], label="Validation Loss", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training and Validation Loss")
ax.legend()
plt.tight_layout()
save_fig(fig, "fig3_training_validation_loss")
plt.show()

In [ ]:
base_model.trainable = True
for layer in base_model.layers:
    if not layer.name.startswith("block5"):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="categorical_crossentropy",
              metrics=["accuracy"])


FT_EPOCHS = 8
start_time = time.time()
history_ft = model.fit(x_train, y_train,
                        validation_data=(x_test, y_test),
                        epochs=FT_EPOCHS,
                        batch_size=BATCH_SIZE)
train_time_phase2 = time.time() - start_time
print(f"\nPhase-2 (fine-tuning) training time: {train_time_phase2:.1f} seconds")

total_train_time = train_time_phase1 + train_time_phase2

In [ ]:
acc_before_ft = history.history["val_accuracy"][-1]
acc_after_ft = history_ft.history["val_accuracy"][-1]

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Before Fine-Tuning", "After Fine-Tuning"],
              [acc_before_ft, acc_after_ft], width=0.5)
ax.set_ylabel("Validation Accuracy")
ax.set_title("Effect of Fine-Tuning on Accuracy")
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
            f"{b.get_height():.3f}", ha="center")
plt.tight_layout()
save_fig(fig, "fig4_finetuning_comparison")
plt.show()

print(f"Validation accuracy before fine-tuning: {acc_before_ft:.4f}")
print(f"Validation accuracy after fine-tuning : {acc_after_ft:.4f}")

In [ ]:
y_pred_probs = model.predict(x_test, batch_size=BATCH_SIZE)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

test_accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="macro")
recall = recall_score(y_true, y_pred, average="macro")
f1 = f1_score(y_true, y_pred, average="macro")

print(f"Test Accuracy     : {test_accuracy:.4f}")
print(f"Precision (macro) : {precision:.4f}")
print(f"Recall (macro)    : {recall:.4f}")
print(f"F1-score (macro)  : {f1:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

total_params = model.count_params()
print(f"\nTotal Parameters: {total_params:,}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("Confusion Matrix")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
save_fig(fig, "fig5_confusion_matrix")
plt.show()

In [ ]:
misclassified_idx = np.where(y_pred != y_true)[0]
n_show = min(10, len(misclassified_idx))
sample_idx = np.random.choice(misclassified_idx, size=n_show, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(x_test[idx])
    ax.set_title(f"T:{CLASS_NAMES[y_true[idx]]}\nP:{CLASS_NAMES[y_pred[idx]]}", fontsize=9)
    ax.axis("off")
fig.suptitle("Misclassified Images")
plt.tight_layout()
save_fig(fig, "fig6_misclassified_images")
plt.show()

In [ ]:
print("=" * 50)
print("RESULTS SUMMARY (Section 18.1)")
print("=" * 50)
print(f"Training Accuracy : {history_ft.history['accuracy'][-1]:.4f}")
print(f"Testing Accuracy  : {test_accuracy:.4f}")
print(f"Precision         : {precision:.4f}")
print(f"Recall            : {recall:.4f}")
print(f"F1-score          : {f1:.4f}")
print(f"Training Time     : {total_train_time:.1f} seconds")
print(f"Total Parameters  : {total_params:,}")

In [ ]:
from tensorflow.keras import Model

def build_lenet5():
    inp = keras.Input(shape=(32, 32, 3))
    x = layers.Conv2D(6, 5, activation="tanh", padding="same")(inp)
    x = layers.AveragePooling2D(2)(x)
    x = layers.Conv2D(16, 5, activation="tanh")(x)
    x = layers.AveragePooling2D(2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(120, activation="tanh")(x)
    x = layers.Dense(84, activation="tanh")(x)
    out = layers.Dense(10, activation="softmax")(x)
    m = Model(inp, out, name="LeNet5")
    m.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return m

def build_alexnet_style():


    inp = keras.Input(shape=(32, 32, 3))
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(inp)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(192, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(384, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(256, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(256, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(1024, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(10, activation="softmax")(x)
    m = Model(inp, out, name="AlexNetStyle")
    m.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return m

SCRATCH_EPOCHS = 15
scratch_histories = {}
scratch_times = {}
scratch_params = {}
scratch_test_acc = {}

for name, builder in [("LeNet-5", build_lenet5), ("AlexNet-style", build_alexnet_style)]:
    print(f"\n--- Training {name} from scratch ---")
    m = builder()
    t0 = time.time()
    h = m.fit(x_train, y_train, validation_data=(x_test, y_test),
              epochs=SCRATCH_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
    elapsed = time.time() - t0
    _, acc = m.evaluate(x_test, y_test, verbose=0)
    scratch_histories[name] = h
    scratch_times[name] = elapsed
    scratch_params[name] = m.count_params()
    scratch_test_acc[name] = acc
    print(f"{name}: test accuracy={acc:.4f}, params={m.count_params():,}, time={elapsed:.1f}s")

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

resnet_base = ResNet50(weights="imagenet", include_top=False,
                        input_shape=(IMG_SIZE, IMG_SIZE, 3))
resnet_base.trainable = False

r_inputs = keras.Input(shape=(32, 32, 3))
r = layers.Resizing(IMG_SIZE, IMG_SIZE)(r_inputs)
r = layers.Lambda(lambda img: img * 255.0)(r)
r = layers.Lambda(resnet_preprocess)(r)
r = resnet_base(r, training=False)
r = layers.GlobalAveragePooling2D()(r)
r = layers.Dense(256, activation="relu")(r)
r = layers.Dropout(0.3)(r)
r_outputs = layers.Dense(10, activation="softmax")(r)

resnet_model = keras.Model(r_inputs, r_outputs, name="ResNet50_TransferLearning")
resnet_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                      loss="categorical_crossentropy", metrics=["accuracy"])

t0 = time.time()
resnet_history = resnet_model.fit(x_train, y_train,
                                   validation_data=(x_test, y_test),
                                   epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
resnet_train_time = time.time() - t0
_, resnet_test_acc = resnet_model.evaluate(x_test, y_test, verbose=0)
resnet_params_count = resnet_model.count_params()
print(f"ResNet50 (TL): test accuracy={resnet_test_acc:.4f}, "
      f"params={resnet_params_count:,}, time={resnet_train_time:.1f}s")

In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess

inception_base = InceptionV3(weights="imagenet", include_top=False,
                              input_shape=(IMG_SIZE, IMG_SIZE, 3))
inception_base.trainable = False

g_inputs = keras.Input(shape=(32, 32, 3))
g = layers.Resizing(IMG_SIZE, IMG_SIZE)(g_inputs)
g = layers.Lambda(lambda img: img * 255.0)(g)
g = layers.Lambda(inception_preprocess)(g)
g = inception_base(g, training=False)
g = layers.GlobalAveragePooling2D()(g)
g = layers.Dense(256, activation="relu")(g)
g = layers.Dropout(0.3)(g)
g_outputs = layers.Dense(10, activation="softmax")(g)

inception_model = keras.Model(g_inputs, g_outputs, name="GoogleNet_InceptionV3_TransferLearning")
inception_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                         loss="categorical_crossentropy", metrics=["accuracy"])

t0 = time.time()
inception_history = inception_model.fit(x_train, y_train,
                                         validation_data=(x_test, y_test),
                                         epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
inception_train_time = time.time() - t0
_, inception_test_acc = inception_model.evaluate(x_test, y_test, verbose=0)
inception_params_count = inception_model.count_params()
print(f"GoogleNet/InceptionV3 (TL): test accuracy={inception_test_acc:.4f}, "
      f"params={inception_params_count:,}, time={inception_train_time:.1f}s")

In [ ]:
vgg_val_acc = history.history["val_accuracy"] + history_ft.history["val_accuracy"]
vgg_val_loss = history.history["val_loss"] + history_ft.history["val_loss"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(scratch_histories["LeNet-5"].history["val_accuracy"], label="LeNet-5")
axes[0].plot(scratch_histories["AlexNet-style"].history["val_accuracy"], label="AlexNet-style")
axes[0].plot(vgg_val_acc, label="VGG16 (TL)")
axes[0].plot(inception_history.history["val_accuracy"], label="GoogleNet (TL)")
axes[0].plot(resnet_history.history["val_accuracy"], label="ResNet50 (TL)")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Validation Accuracy")
axes[0].set_title("Validation Accuracy by Architecture")
axes[0].legend(fontsize=9)

axes[1].plot(scratch_histories["LeNet-5"].history["val_loss"], label="LeNet-5")
axes[1].plot(scratch_histories["AlexNet-style"].history["val_loss"], label="AlexNet-style")
axes[1].plot(vgg_val_loss, label="VGG16 (TL)")
axes[1].plot(inception_history.history["val_loss"], label="GoogleNet (TL)")
axes[1].plot(resnet_history.history["val_loss"], label="ResNet50 (TL)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation Loss")
axes[1].set_title("Validation Loss by Architecture")
axes[1].legend(fontsize=9)

plt.tight_layout()
save_fig(fig, "fig7_architecture_comparison")
plt.show()

In [ ]:
comparison_table = [
    ("LeNet-5",       scratch_params["LeNet-5"],       scratch_test_acc["LeNet-5"],       scratch_times["LeNet-5"]),
    ("AlexNet-style",  scratch_params["AlexNet-style"], scratch_test_acc["AlexNet-style"], scratch_times["AlexNet-style"]),
    ("VGG16 (TL)",     total_params,                    acc_after_ft,                      total_train_time),
    ("GoogleNet (TL)", inception_params_count,          inception_test_acc,                inception_train_time),
    ("ResNet50 (TL)",  resnet_params_count,             resnet_test_acc,                   resnet_train_time),
]

print(f"{'Model':<16}{'Parameters':>14}{'Accuracy (%)':>16}{'Time (s)':>12}")
print("-" * 58)
for name, params, acc, t in comparison_table:
    print(f"{name:<16}{params:>14,}{acc*100:>15.2f}{t:>12.1f}")

In [ ]:
SUBSET_SIZE = 8000
rng = np.random.RandomState(0)
sub_idx = rng.choice(len(x_train), SUBSET_SIZE, replace=False)
test_sub_idx = rng.choice(len(x_test), 1500, replace=False)
x_sub, y_sub = x_train[sub_idx], y_train[sub_idx]
x_test_sub, y_test_sub = x_test[test_sub_idx], y_test[test_sub_idx]

BASELINE = {"lr": 0.001, "batch_size": 32, "epochs": 10,
            "optimizer": "adam", "dense_units": 256, "frozen": "all"}

def build_variant(dense_units, frozen):
    vb = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    vb.trainable = (frozen != "all")
    if frozen == "partial":
        for layer in vb.layers:
            if not layer.name.startswith("block5"):
                layer.trainable = False
    vin = keras.Input(shape=(32, 32, 3))
    v = layers.Resizing(IMG_SIZE, IMG_SIZE)(vin)
    v = layers.Lambda(lambda img: img * 255.0)(v)
    v = layers.Lambda(preprocess_input)(v)
    v = vb(v, training=False)
    v = layers.GlobalAveragePooling2D()(v)
    v = layers.Dense(dense_units, activation="relu")(v)
    v = layers.Dropout(0.3)(v)
    vout = layers.Dense(10, activation="softmax")(v)
    return keras.Model(vin, vout)

def run_variant(tag, lr, batch_size, epochs, optimizer, dense_units, frozen):
    m = build_variant(dense_units, frozen)
    opt = (keras.optimizers.Adam(learning_rate=lr) if optimizer == "adam"
           else keras.optimizers.SGD(learning_rate=lr))
    m.compile(optimizer=opt, loss="categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    h = m.fit(x_sub, y_sub, validation_data=(x_test_sub, y_test_sub),
              epochs=epochs, batch_size=batch_size, verbose=0)
    elapsed = time.time() - t0
    return {"tag": tag, "val_accuracy": h.history["val_accuracy"][-1], "time": elapsed}


sweep = []
for lr in [0.001, 0.0001]:
    cfg = {**BASELINE, "lr": lr}
    sweep.append((f"Learning Rate={lr}", cfg))
for bs in [16, 32, 64]:
    cfg = {**BASELINE, "batch_size": bs}
    sweep.append((f"Batch Size={bs}", cfg))
for ep in [10, 20]:
    cfg = {**BASELINE, "epochs": ep}
    sweep.append((f"Epochs={ep}", cfg))
for opt_name in ["adam", "sgd"]:
    cfg = {**BASELINE, "optimizer": opt_name}
    sweep.append((f"Optimizer={opt_name}", cfg))
for du in [128, 256]:
    cfg = {**BASELINE, "dense_units": du}
    sweep.append((f"Dense Units={du}", cfg))
for frozen in ["all", "partial"]:
    cfg = {**BASELINE, "frozen": frozen}
    if frozen == "partial":
        cfg["lr"] = 1e-5
    sweep.append((f"Frozen Layers={frozen}", cfg))

hp_results = []
for tag, cfg in sweep:
    print(f"Running: {tag} ...")
    result = run_variant(tag, cfg["lr"], cfg["batch_size"], cfg["epochs"],
                          cfg["optimizer"], cfg["dense_units"], cfg["frozen"])
    hp_results.append(result)
    print(f"  -> val_accuracy={result['val_accuracy']:.4f}, time={result['time']:.1f}s")

print(f"\n{'Setting':<24}{'Val Accuracy':>14}{'Time (s)':>12}")
print("-" * 50)
for r in hp_results:
    print(f"{r['tag']:<24}{r['val_accuracy']:>14.4f}{r['time']:>12.1f}")